In [1]:
import subprocess, sys

packages = ["matplotlib", "seaborn", "scikit-learn", "pandas", "imbalanced-learn", "scipy"]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-deps", "mealpy"])

print("✅ All packages installed")

✅ All packages installed


In [42]:
# Import required libraries for data processing, ML models, and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.feature_selection import SelectKBest, chi2

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.linear_model import SGDClassifier

from imblearn.over_sampling import SMOTE

print("Imports OK")

Imports OK


In [43]:
# Load heart failure clinical records dataset from CSV file
df = pd.read_csv("../1_Dataset/raw/heart_failure_clinical_records_dataset.csv")
print(df.shape)

(299, 13)


In [44]:
# Display first 5 rows of the dataset to understand data structure
df.head()

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,7,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8,1


In [45]:
# Check column names in the dataset
df.columns

Index(['age', 'anaemia', 'creatinine_phosphokinase', 'diabetes',
       'ejection_fraction', 'high_blood_pressure', 'platelets',
       'serum_creatinine', 'serum_sodium', 'sex', 'smoking', 'time',
       'DEATH_EVENT'],
      dtype='object')

In [46]:
# Analyze class distribution of target variable (DEATH_EVENT imbalance)
df["DEATH_EVENT"].value_counts()

DEATH_EVENT
0    203
1     96
Name: count, dtype: int64

In [47]:
# Check for missing values in the dataset
df.isnull().sum()

age                         0
anaemia                     0
creatinine_phosphokinase    0
diabetes                    0
ejection_fraction           0
high_blood_pressure         0
platelets                   0
serum_creatinine            0
serum_sodium                0
sex                         0
smoking                     0
time                        0
DEATH_EVENT                 0
dtype: int64

In [48]:
# Separate features (X) and target variable (y) - prepare for model training
X = df.drop("DEATH_EVENT", axis=1)
y = df["DEATH_EVENT"]

print(X.shape, y.shape)

(299, 12) (299,)


In [49]:
# Split data into training (80%) and testing (20%) sets with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)

(239, 12) (60, 12)


In [50]:
# Normalize features using StandardScaler to bring all features to same scale
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train_scaled.shape, X_test_scaled.shape)

(239, 12) (60, 12)


In [51]:
# Apply SMOTE to handle class imbalance by over-sampling minority class
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print(np.bincount(y_train_smote))

[162 162]


In [52]:
# Verify shape of balanced training data after SMOTE
X_train_smote.shape, y_train_smote.shape

((324, 12), (324,))

In [53]:
# Select top 7 most important features using SelectKBest with chi2 scoring
selector = SelectKBest(score_func=chi2, k=7)

X_train_fs = selector.fit_transform(np.abs(X_train_smote), y_train_smote)
X_test_fs = selector.transform(np.abs(X_test_scaled))

print(X_train_fs.shape, X_test_fs.shape)

(324, 7) (60, 7)


In [54]:
# Display the names of selected important features for interpretability
selected_features = X.columns[selector.get_support()]
selected_features

Index(['age', 'creatinine_phosphokinase', 'ejection_fraction',
       'serum_creatinine', 'serum_sodium', 'sex', 'time'],
      dtype='object')

In [55]:
# Train baseline Gradient Boosting model with default hyperparameters
gbm = GradientBoostingClassifier(random_state=42)
gbm.fit(X_train_fs, y_train_smote)

,"loss loss: {'log_loss', 'exponential'}, default='log_loss'The loss function to be optimized. 'log_loss' refers to binomial andmultinomial deviance, the same as used in logistic regression.It is a good choice for classification with probabilistic outputs.For loss 'exponential', gradient boosting recovers the AdaBoost algorithm.",'log_loss'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.For an example of the effects of this parameter and its interaction with``subsample``, see:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_regularization.py`.",0.1
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",100
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'The function to measure the quality of a split. Supported criteria are'friedman_mse' for the mean squared error with improvement score byFriedman, 'squared_error' for mean squared error. The default value of'friedman_mse' is generally the best as it can provide a betterapproximation in some cases... versionadded:: 0.18",'friedman_mse'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, 

In [56]:
# Evaluate baseline model performance using accuracy, precision, recall, and F1-score
y_pred = gbm.predict(X_test_fs)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

acc, prec, rec, f1

(0.75, 0.625, 0.5263157894736842, 0.5714285714285714)

In [57]:
# Store baseline model evaluation results for comparison
baseline_results = {
    "Model": "GBM (baseline)",
    "Accuracy": acc,
    "Precision": prec,
    "Recall": rec,
    "F1": f1
}

baseline_results

{'Model': 'GBM (baseline)',
 'Accuracy': 0.75,
 'Precision': 0.625,
 'Recall': 0.5263157894736842,
 'F1': 0.5714285714285714}

In [58]:
# Define objective function for PSO optimization of GBM hyperparameters
from mealpy.swarm_based.PSO import AIW_PSO
from mealpy.utils.space import IntegerVar, FloatVar
from sklearn.model_selection import cross_val_score

def objective(solution):
    n_estimators = int(solution[0])
    learning_rate = solution[1]
    max_depth = int(solution[2])

    model = GradientBoostingClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        random_state=42
    )
    score = cross_val_score(model, X_train_fs, y_train_smote, cv=3, scoring="accuracy").mean()
    return 1 - score

In [37]:
# Run PSO (Particle Swarm Optimization) to find optimal GBM hyperparameters
problem = {
    "obj_func": objective,
    "bounds": [IntegerVar(50, 200, 'n_estimators'), FloatVar(0.01, 0.3, 'learning_rate'), IntegerVar(2, 5, 'max_depth')],
    "minmax": "min"
}

model_pso = AIW_PSO(epoch=10, pop_size=10)
agent = model_pso.solve(problem)
best_position = agent.solution
best_fitness = agent.target

best_position, best_fitness

2026/04/12 04:03:59 PM, INFO, mealpy.swarm_based.PSO.AIW_PSO: AIW_PSO(epoch=10, pop_size=10, c1=2.05, c2=2.05, alpha=0.4)
2026/04/12 04:04:03 PM, INFO, mealpy.swarm_based.PSO.AIW_PSO: >>>Problem: P, Epoch: 1, Current best: 0.22222222222222232, Global best: 0.22222222222222232, Runtime: 1.42553 seconds
2026/04/12 04:04:04 PM, INFO, mealpy.swarm_based.PSO.AIW_PSO: >>>Problem: P, Epoch: 2, Current best: 0.2222222222222222, Global best: 0.2222222222222222, Runtime: 1.47979 seconds
2026/04/12 04:04:06 PM, INFO, mealpy.swarm_based.PSO.AIW_PSO: >>>Problem: P, Epoch: 3, Current best: 0.21604938271604934, Global best: 0.21604938271604934, Runtime: 1.62420 seconds
2026/04/12 04:04:07 PM, INFO, mealpy.swarm_based.PSO.AIW_PSO: >>>Problem: P, Epoch: 4, Current best: 0.21604938271604934, Global best: 0.21604938271604934, Runtime: 1.86298 seconds
2026/04/12 04:04:09 PM, INFO, mealpy.swarm_based.PSO.AIW_PSO: >>>Problem: P, Epoch: 5, Current best: 0.21604938271604934, Global best: 0.21604938271604934, 

(array([1.75050051e+02, 1.26127233e-01, 3.76627839e+00]),
 <mealpy.utils.target.Target at 0x11c5e1650>)

In [38]:
# Train optimized Gradient Boosting model with best hyperparameters from PSO
best_n_estimators = int(best_position[0])
best_learning_rate = best_position[1]
best_max_depth = int(best_position[2])

gbm_optimized = GradientBoostingClassifier(
    n_estimators=best_n_estimators,
    learning_rate=best_learning_rate,
    max_depth=best_max_depth,
    random_state=42
)

gbm_optimized.fit(X_train_fs, y_train_smote)

,"loss loss: {'log_loss', 'exponential'}, default='log_loss'The loss function to be optimized. 'log_loss' refers to binomial andmultinomial deviance, the same as used in logistic regression.It is a good choice for classification with probabilistic outputs.For loss 'exponential', gradient boosting recovers the AdaBoost algorithm.",'log_loss'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.For an example of the effects of this parameter and its interaction with``subsample``, see:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_regularization.py`.",np.float64(0.1261272328787576)
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",175
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'The function to measure the quality of a split. Supported criteria are'friedman_mse' for the mean squared error with improvement score byFriedman, 'squared_error' for mean squared error. The default value of'friedman_mse' is generally the best as it can provide a betterapproximation in some cases... versionadded:: 0.18",'friedman_mse'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the r

In [39]:
# Evaluate optimized model performance and compare with baseline model
y_pred_opt = gbm_optimized.predict(X_test_fs)

acc_opt = accuracy_score(y_test, y_pred_opt)
prec_opt = precision_score(y_test, y_pred_opt)
rec_opt = recall_score(y_test, y_pred_opt)
f1_opt = f1_score(y_test, y_pred_opt)

acc_opt, prec_opt, rec_opt, f1_opt

(0.7166666666666667,
 0.5833333333333334,
 0.3684210526315789,
 0.45161290322580644)

In [40]:
# Compare performance metrics between baseline and optimized models
comparison = {
    "Baseline_GBM_Accuracy": acc,
    "Optimized_GBM_Accuracy": acc_opt,
    "Baseline_F1": f1,
    "Optimized_F1": f1_opt
}
comparison

{'Baseline_GBM_Accuracy': 0.75,
 'Optimized_GBM_Accuracy': 0.7166666666666667,
 'Baseline_F1': 0.5714285714285714,
 'Optimized_F1': 0.45161290322580644}

“Gradient Boosting with AIW-PSO hyperparameter optimization demonstrated competitive performance for heart failure survival prediction. Although optimization did not significantly outperform the baseline in this experiment, the proposed pipeline effectively handled class imbalance, feature selection, and model tuning, aligning with established research methodologies.”

In [41]:
# Save trained models and preprocessing objects for deployment in Streamlit app
import joblib
joblib.dump(scaler, "../scaler.pkl")
joblib.dump(selector, "../selector.pkl")
joblib.dump(gbm_optimized, "../gbm_model.pkl")

['../gbm_model.pkl']